# Домашнее задание: Введение в LLM**Занятие 41 | Неделя 21**## ЗаданиеВыполните все ячейки с пометкой `# ВАШ КОД ЗДЕСЬ`.Ответьте на теоретические вопросы в markdown-ячейках.**Дедлайн:** до следующего занятия.Загрузите ноутбук в LMS.**Критерии:**- Код работает на Colab T4 GPU- Все ячейки выполнены, результаты видны- Теоретические вопросы отвечены своими словами (не копипаст)

---## 0. ПодготовкаRuntime -> Change runtime type -> **T4 GPU**

In [2]:
import torch
import time
print(f'PyTorch: {torch.__version__}')
print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
  print(f'GPU: {torch.cuda.get_device_name(0)}')
  device = torch.device('cuda')
else:
  device = torch.device('cpu')
  print(f'Device: {device}')

PyTorch: 2.10.0+cu128
CUDA: True
GPU: Tesla T4


In [4]:
!pip install transformers accelerate bitsandbytes sentencepiece -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 13.8 MB/s eta 0:00:00


In [3]:
from transformers import pipeline, AutoTokenizer
import numpy as np
import pandas as pd
import torch
import logging
import warnings
from transformers import logging as transformers_logging

---## Задание 1: Токенизация (Слайд 9)Загрузите токенизатор модели `Qwen/Qwen2.5-0.5B-Instruct`.Токенизируйте 10 предложений: по 3-4 на каждом языке (русский, казахский, английский).Выведите для каждого: текст, количество токенов, сами токены.Постройте таблицу: язык, среднее кол-во токенов, среднее кол-во символов, отношение.

In [5]:
# 10 предложений: 3-4 на EN, 3-4 на RU, 3 на KZ
# Подберите предложения примерно одинакового смысла на разных языках

tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-0.5B-Instruct")

sentences = {
    "EN": [
        "Astana is the capital of Kazakhstan.",
        "The weather here can change quickly.",
        "I enjoy walking along the river.",
        "This city is modern and beautiful."
    ],
    "RU": [
        "Астана — столица Казахстана.",
        "Погода здесь может быстро меняться.",
        "Мне нравится гулять вдоль реки.",
        "Этот город современный и красивый."
    ],
    "KZ": [
        "Астана — Қазақстанның астанасы.",
        "Мұнда ауа райы тез өзгереді.",
        "Мен өзен бойымен серуендегенді ұнатамын."
    ]
}

rows = []

for lang, texts in sentences.items():
    for text in texts:
        tokens = tokenizer.tokenize(text)
        rows.append({
            "Language": lang,
            "Text": text,
            "Tokens": ", ".join(tokens),
            "Num Tokens": len(tokens),
            "Num Chars": len(text)
        })

df = pd.DataFrame(rows)

print("=== Detailed Table ===")
print(df.to_string(index=False))

summary = (
    df.groupby("Language")
    .agg({
        "Num Tokens": "mean",
        "Num Chars": "mean"
    })
    .rename(columns={
        "Num Tokens": "Avg Tokens",
        "Num Chars": "Avg Chars"
    })
)

summary["Token/Char Ratio"] = summary["Avg Tokens"] / summary["Avg Chars"]

print("\n=== Summary Table ===")
print(summary.round(3).to_string())

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

=== Detailed Table ===
Language                                     Text                                                                                                                  Tokens  Num Tokens  Num Chars
      EN     Astana is the capital of Kazakhstan.                                                                      Ast, ana, Ġis, Ġthe, Ġcapital, Ġof, ĠKazakhstan, .           8         36
      EN     The weather here can change quickly.                                                                        The, Ġweather, Ġhere, Ġcan, Ġchange, Ġquickly, .           7         36
      EN         I enjoy walking along the river.                                                                            I, Ġenjoy, Ġwalking, Ġalong, Ġthe, Ġriver, .           7         32
      EN       This city is modern and beautiful.                                                                          This, Ġcity, Ġis, Ġmodern, Ġand, Ġbeautiful, .           7         34
      RU    

**Вопрос 1.1 (Слайд 9):** Почему казахский и русский текст требуют больше токенов, чем английский? Как это влияет на стоимость использования LLM?*Ваш ответ:*

Большинство моделей обучались на английских текстах, из-за чего отдельные английские слова представляют из себя токены, в то время как более редкие русские и казахские слова могут состоять из нескольких токенов потому что встречались реже. В результате на это уходит в 2-2.5 раза больше токенов.

**Вопрос 1.2 (Слайд 9):** Что такое BPE (Byte Pair Encoding)? Опишите алгоритм в 3-4 шагах своими словами.*Ваш ответ:*

---## Задание 2: Генерация текста (Слайды 8, 13)Загрузите модель через pipeline и сгенерируйте текст с разными параметрами.

In [10]:
# Загрузите модель
# Задание: сгенерируйте ответ на один и тот же промпт
# при температурах: 0.1, 0.5, 0.7, 1.0, 1.5
# Промпт: "Расскажи интересный факт об Астане"
# Для каждой температуры — выведите температуру и ответ

warnings.filterwarnings("ignore")
transformers_logging.set_verbosity_error()

generator = pipeline(
    "text-generation",
    model="Qwen/Qwen2.5-0.5B-Instruct",
    device_map="auto",
    model_kwargs={"dtype": torch.float16}
)

prompt = "Расскажи интересный факт об Астане"

messages = [
    {"role": "system", "content": "Ты полезный помощник."},
    {"role": "user", "content": prompt}
]

formatted_prompt = generator.tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

temps = [0.1, 0.5, 0.7, 1.0, 1.5]

for t in temps:
    result = generator(
        formatted_prompt,
        max_new_tokens=250,
        temperature=t,
        do_sample=True if t > 0 else False,
        return_full_text=False,
        pad_token_id=generator.tokenizer.eos_token_id
    )

    print(f"\n--- Temperature: {t} ---")
    print(result[0]["generated_text"].strip())

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]


--- Temperature: 0.1 ---
В Астане есть несколько уникальных мест, которые необычно сочетают в себе исторический и современный национальный дух:

1. **Астана-Кабул**: Это самый старый город в мире, основан в 780 году нашей эры. Он был основан как царством Ашрафа I.

2. **Санкт-Петербург**: Хотя Санкт-Петербург является столицей России, он также известен своими историческими достопримечательностями, таких как Кремль, Собор Николая Великого и Музейный зал.

3. **Москва**: Хотя Москва известна своей культурой и историей, она также имеет множество уникальных мест, таких как:

   - **Большая Католическая церковь (Католическая крепость)**: Это один из самых древних и крупнейших храмов в мире.
   
   - **Памятники князю Дмитрию**: Здесь можно найти множество архитектурных памятников, включая Памятник к

--- Temperature: 0.5 ---
Вот интересный факт о Астане:

Астрахань - это самая большая и старейшая республика России. Она была основана в 1396 году, когда Павел Калмыков приказал построить церк

**Вопрос 2.1 (Слайд 13):** Как температура влияет на генерацию? При какой температуре ответы наиболее стабильные? При какой — начинается "мусор"?
*Ваш ответ:*

При температуре 0 модель выдаёт максимально подтверждённую и вероятную информацию.

Температура 0.7 является балансом между вариативностью и верностью информации.

При температуре 1.5 модель наиболее сильно разнообразит ответ, но может из-за этого галлюцинировать и выдумывать информацию.


In [14]:
# Задание: запустите генерацию 5 раз при T=0 и 5 раз при T=0.9
# Промпт: "Придумай слоган для казахстанского IT-университета"
# Выведите все ответы и объясните, почему при T=0 они одинаковые

prompt = "Придумай слоган для казахстанского IT-университета"

print("=== T=0 ===")
for _ in range(5):
    result = generator(
        prompt,
        max_new_tokens=200,
        temperature=0,
        do_sample=False
    )
    print(result[0]["generated_text"])

print("\n=== T=0.9 ===")
for i in range(5):
    result = generator(
      prompt,
      max_new_tokens=200,
      temperature=0.9,
      do_sample=True,
      return_full_text=False,
      pad_token_id=generator.tokenizer.eos_token_id
    )

    print(f"Вариант {i+1}: {result[0]['generated_text'].strip()}")
    print("-" * 30)

=== T=0 ===
Придумай слоган для казахстанского IT-университета "Казахстанский Институт Технологий". 

Вот список ключевых слов и их значения:

1. Казахстан
2. Институт
3. Технологии

Слово "Казахстан" в данном контексте означает:
- Страну, где находится университет
- Страну, где он расположен

Слово "Институт" в данном контексте означает:
- Университет, который находится в стране
- Университет, который расположен в стране

Слово "Технологии" в данном контексте означает:
- Производство или создание технологических инноваций
- Создание или внедрение новых технологий

Слово "Казахстанский" в данном контексте означает:
- Страна,
Придумай слоган для казахстанского IT-университета "Казахстанский Институт Технологий". 

Вот список ключевых слов и их значения:

1. Казахстан
2. Институт
3. Технологии

Слово "Казахстан" в данном контексте означает:
- Страну, где находится университет
- Страну, где он расположен

Слово "Институт" в данном контексте означает:
- Университет, который находится в стр

**Вопрос 2.2 (Слайд 8):** Что такое autoregressive генерация? Почему модель генерирует токен за токеном, а не весь текст сразу?*Ваш ответ:*

---## Задание 3: Zero-shot vs Few-shot (Слайд 14)Реализуйте классификатор тональности отзывов двумя способами:zero-shot (без примеров) и few-shot (с примерами в промпте).

In [18]:
# Задание: напишите функцию zero-shot классификации
# Вход: текст отзыва
# Выход: "позитивный", "негативный" или "нейтральный"
# Используйте system prompt для описания задачиdef classify_zero_shot(text):

def classify_zero_shot(text):
    messages = [
        {"role": "system", "content": "Ты — классификатор тональности. Отвечай только одним словом: позитивный, негативный или нейтральный."},
        {"role": "user", "content": f"Определи тональность текста: {text}"}
    ]

    formatted_prompt = generator.tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )

    result = generator(
        formatted_prompt,
        max_new_tokens=15,
        temperature=0,
        do_sample=False,
        return_full_text=False,
        pad_token_id=generator.tokenizer.eos_token_id
    )

    answer = result[0]["generated_text"].strip().lower()

    if "позитив" in answer:
        return "позитивный"
    if "негатив" in answer:
        return "негативный"
    if "нейтрал" in answer:
        return "нейтральный"

    return f"не определено ({answer})"

test_reviews = [
    "Все было идеально, спасибо большое!",
    "Никогда больше не вернусь в это место.",
    "Обычное кафе, ничего выдающегося.",
    "Быстрая доставка, товар как на фото.",
    "Обман чистой воды, не заказывайте.",
    "Сойдет, но за такую цену ожидал большего.",
]

for review in test_reviews:
    print(f"Отзыв: {review}")
    print(f"Классификация: {classify_zero_shot(review)}")
    print("-" * 50)

Отзыв: Все было идеально, спасибо большое!
Классификация: позитивный
--------------------------------------------------
Отзыв: Никогда больше не вернусь в это место.
Классификация: не определено (negativity)
--------------------------------------------------
Отзыв: Обычное кафе, ничего выдающегося.
Классификация: негативный
--------------------------------------------------
Отзыв: Быстрая доставка, товар как на фото.
Классификация: позитивный
--------------------------------------------------
Отзыв: Обман чистой воды, не заказывайте.
Классификация: негативный
--------------------------------------------------
Отзыв: Сойдет, но за такую цену ожидал большего.
Классификация: негативный
--------------------------------------------------


In [21]:
# Задание: напишите функцию few-shot классификации
# Добавьте 4-5 примеров в system promptdef classify_few_shot(text):


def classify_few_shot(text):
    system_content = (
        "Ты — эксперт по анализу тональности отзывов. "
        "Классифицируй текст как: позитивный, негативный или нейтральный. "
        "\nПримеры:\n"
        "Текст: 'Потрясающее место, вернемся еще раз!'\nОтвет: позитивный\n"
        "Текст: 'Все ужасно, еда холодная.'\nОтвет: негативный\n"
        "Текст: 'Обычный магазин, ничего особенного.'\nОтвет: нейтральный\n"
        "Текст: 'Доставили быстро, но товар поврежден.'\nОтвет: нейтральный\n"
        "Текст: 'Очень вежливый персонал и чисто.'\nОтвет: позитивный"
    )

    messages = [
        {"role": "system", "content": system_content},
        {"role": "user", "content": f"Текст: '{text}'\nОтвет:"}
    ]

    formatted_prompt = generator.tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )

    result = generator(
        formatted_prompt,
        max_new_tokens=10,
        temperature=0,
        do_sample=False,
        return_full_text=False,
        pad_token_id=generator.tokenizer.eos_token_id
    )

    answer = result[0]["generated_text"].strip().lower()

    if "позитив" in answer: return "позитивный"
    if "негатив" in answer: return "негативный"
    if "нейтрал" in answer: return "нейтральный"

    return answer

test_reviews = [
    "Все было идеально, спасибо большое!",
    "Никогда больше не вернусь в это место.",
    "Обычное кафе, ничего выдающегося.",
    "Быстрая доставка, товар как на фото.",
    "Обман чистой воды, не заказывайте.",
    "Сойдет, но за такую цену ожидал большего.",
]

print(f"{'Отзыв':<45} | {'Результат'}")
print("-" * 65)
for review in test_reviews:
    res = classify_few_shot(review)
    print(f"{review[:43]:<45} | {res}")

Отзыв                                         | Результат
-----------------------------------------------------------------
Все было идеально, спасибо большое!           | позитивный
Никогда больше не вернусь в это место.        | негативный
Обычное кафе, ничего выдающегося.             | негативный
Быстрая доставка, товар как на фото.          | позитивный
Обман чистой воды, не заказывайте.            | негативный
Сойдет, но за такую цену ожидал большего.     | негативный


**Вопрос 3.1 (Слайд 14):** В чем разница между zero-shot и few-shot? Когда few-shot дает лучшие результаты?*Ваш ответ:*

---## Задание 4: System Prompt (Слайд 14)Напишите 3 разных system prompt для одной модели, чтобы она вела себя как:1. Учитель математики2. Переводчик каз-рус3. Ваша идея (придумайте свой вариант)

In [19]:
# Задание: 3 system prompt + по 2 тестовых запроса на каждый
# Выведите ответы модели для каждого варианта

system_prompt_math = "Ты учитель математики. Объясняй просто и понятно."

queries = ["Что такое производная?", "Объясни дроби"]

for q in queries:
    print(generator(system_prompt_math + "\n" + q)[0]["generated_text"])

Ты учитель математики. Объясняй просто и понятно.
Что такое производная? Она какая важность в математике? Как она работает?
Давайте рассмотрим пример, чтобы понять её действия лучше.

При каком условии производная существует? Какова формула для её определения?

Конечно, давайте поговорим подробнее о производной:

1. **Производная** - это функция, которая задает её производную. Это вычисление непрерывного изменения функции при изменении одного из ее параметров.

2. **Существование производной**:
   - Если функция не дифференцируема (не имеет непрерывных производных), то её производная не существовала.
   - Если функция дифференцируется (такие, что её производная существует), то её производная определена и называется производной.

3. **Формула для определения производной**:
   - Для линейной функции $f(x)$ производная равна значению её функции на определённом интервале.
   - Для квадратичной функции $ax^2 + bx + c$
Ты учитель математики. Объясняй просто и понятно.
Объясни дроби как числа

**Вопрос 4.1 (Слайд 14):** Зачем нужен system prompt? Чем он отличается от обычного user-сообщения?*Ваш ответ:*

---## Задание 5: Теоретические вопросы (по слайдам)Ответьте на вопросы своими словами. Не копируйте — объясните как вы понимаете.

**Вопрос 5.1 (Слайд 6):** Чем отличается Self-Attention в BERT (bidirectional) от Causal Attention в GPT (unidirectional)? Нарисуйте или опишите маску внимания для обоих случаев.*Ваш ответ:*

В BERT используется двунаправленное внимание (bidirectional) — каждое слово «смотрит» на все остальные слова в предложении (и слева, и справа). В GPT используется каузальное внимание (unidirectional) — каждое слово видит только предыдущие слова (слева направо).

Маска BERT:

1 1 1 1

1 1 1 1

1 1 1 1

1 1 1 1

Маска GPT:

1 0 0 0

1 1 0 0

1 1 1 0

1 1 1 1

**Вопрос 5.2 (Слайд 7):** Назовите 3 типа архитектур LLM (encoder-only, decoder-only, encoder-decoder). Для каждого приведите пример модели и задачу, для которой он подходит.*Ваш ответ:*

3 типа архитектур LLM:

- Encoder-only
  - Пример: BERT
  - Подходит для: классификации текста, поиска, анализа
  - Почему: понимает текст, но не генерирует

- Decoder-only
  - Пример: GPT
  - Подходит для: генерации текста, чат-ботов
  - Почему: генерирует токены по одному

- Encoder-Decoder
  - Пример: T5
  - Подходит для: переводов, суммаризации
  - Почему: сначала понимает вход, потом генерирует ответ

**Вопрос 5.3 (Слайд 11):** Что значит "открытая модель"? Объясните разницу между API-only, Open Weights и Open Source. Почему открытые модели важны для Казахстана?*Ваш ответ:*

Открытая модель это API-only. Сама модель закрыта, доступ только через API

Пример: нельзя скачать, только отправлять запросы

У Open Weights можно скачать веса модели и запускать локально, но код может быть не полностью открыт

В Open Source всё открыто: код, веса, обучение. Можно полностью воспроизвести модель.

Для Казахстана важно то, что можно запускать модели локально, лучше для безопасности данных и можно адаптировать под казахский и русский языки.

**Вопрос 5.4 (Слайд 12):** Что такое квантизация? Зачем она нужна? Сколько VRAM нужно для LLaMA 3 8B в FP16 и в INT4?*Ваш ответ:*

Квантизация это уменьшение точности чисел, то есть веса занимают меньше памяти.
Это уменьшает использование VRAM, ускоряет модель и позволяет запускать модель на слабом железе.

Пример (LLaMA 3 8B):

FP16: ~16 GB VRAM

INT4: ~4–5 GB VRAM

Разница примерно в 3–4 раза

**Вопрос 5.5 (Слайд 16):** Назовите 3 ограничения LLM. Для каждого приведите конкретный пример, когда это может быть проблемой.*Ваш ответ:*

1. Галлюцинации (ошибки), модель может придумывать факты

Пример: выдуманный источник или неправильная дата

2. Зависимость от данных обучения.
Если данных мало или они плохие, то получаются плохие ответы.

Пример: слабое знание казахского языка

3. Нет настоящего понимания. Модель не «думает», а предсказывает слова

Пример: может ошибаться в логике или математике

---## Чеклист перед сдачей- [ ] Задание 1: токенизация 10 предложений + таблица- [ ] Задание 2: генерация при 5 температурах + анализ- [ ] Задание 3: zero-shot и few-shot классификация + сравнение- [ ] Задание 4: 3 system prompt + тестовые запросы- [ ] Задание 5: все 5 теоретических вопросов отвечены- [ ] Все вопросы отвечены своими словами- [ ] Ноутбук запускается с нуля (Runtime -> Restart and Run All)**Загрузите ноутбук в LMS.**